# Galileo Brain v9 — Training Definitivo

Fine-tuning del modello Qwen3.5-2B su dataset v9:
- **126,280 training** examples (31 sezioni, 24 action tags, 22 componenti)
- **350 eval** examples (50 per intent, bilanciati)

### Miglioramenti v9 vs v7
- **LoRA r=16** (vs r=64) — SOTA per routing/classificazione
- **Q5_K_M** GGUF (vs q4_k_m) — qualita' superiore, M1 8GB ha headroom
- **Completions-only training** — il modello impara SOLO a generare JSON
- **Packing manuale** — riduce steps 10x (da 22h a ~2h)
- **24/24 action tags** coperti (vs ~10/16 in v7)
- **7.1% vision** (vs 2.9% in v7)
- **173 LLM hints unici** (vs 85 in v7)

### Deploy
- Export GGUF q5_k_m → Ollama locale su M1 8GB (~1.5GB)
- `ollama create galileo-brain -f Modelfile`

In [ ]:
# ═══ Cell 1: Installazione ═══
import sys
if 'google.colab' in sys.modules:
    !pip install --no-deps trl peft accelerate bitsandbytes
    !pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
    !pip install --no-deps xformers triton
    !pip install 'transformers>=5.2.0,<=5.3.0' 'trl==0.22.2' datasets

print('✅ Installazione completata!')
print('⚠️  ORA RIAVVIA IL RUNTIME: Runtime → Riavvia runtime')
print('   Poi esegui dalla Cell 2 in poi')

In [ ]:
# ═══ Cell 2: Caricamento modello Qwen3.5-2B ═══
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'unsloth/Qwen3.5-2B'
LOAD_4BIT = False  # bf16 LoRA (raccomandato SOTA)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=LOAD_4BIT,
)

print(f'\n✅ Modello caricato: {MODEL_NAME}')
print(f'   bf16 LoRA | Device: {model.device}')
print(f'   VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

In [ ]:
# ═══ Cell 3: LoRA r=16 (SOTA per routing) ═══
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # v9: r=16 (SOTA per classificazione, r=64 era overkill)
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=32,  # alpha/r = 2 (raccomandato SOTA)
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA r=16 configurato!')
print(f'   Parametri trainabili: {trainable:,} / {total_params:,} ({100*trainable/total_params:.2f}%)')
print(f'   VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

In [ ]:
# ═══ Cell 4: Upload dataset v9 ═══
# Opzione A: Upload diretto (lento per 126K)
# from google.colab import files
# uploaded = files.upload()

# Opzione B: Google Drive (raccomandato per 126K)
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
DRIVE_DIR = '/content/drive/MyDrive/galileo-brain-v9-training'

train_file = None
eval_file = None

# Cerca nella directory Drive
if os.path.exists(DRIVE_DIR):
    for f in os.listdir(DRIVE_DIR):
        if 'v9' in f and f.endswith('.jsonl') and 'eval' not in f:
            shutil.copy(os.path.join(DRIVE_DIR, f), f'/content/{f}')
            train_file = f'/content/{f}'
        elif 'v9' in f and 'eval' in f and f.endswith('.jsonl'):
            shutil.copy(os.path.join(DRIVE_DIR, f), f'/content/{f}')
            eval_file = f'/content/{f}'

if not train_file:
    print('⚠️  Dataset non trovato in Drive!')
    print(f'   Carica galileo-brain-v9.jsonl in: {DRIVE_DIR}')
    print('   Oppure usa upload diretto (decommentare Opzione A)')
else:
    import json
    with open(train_file) as f:
        n_train = sum(1 for _ in f)
    n_eval = 0
    if eval_file:
        with open(eval_file) as f:
            n_eval = sum(1 for _ in f)
    print(f'✅ Train: {train_file} ({n_train:,} esempi)')
    print(f'✅ Eval:  {eval_file} ({n_eval:,} esempi)' if eval_file else '⚠️  No eval file')

In [ ]:
# ═══ Cell 5: Preparazione dataset + TOKENIZER FIX ═══
from datasets import Dataset, load_dataset

# TOKENIZER FIX: Qwen3.5 carica come VL, estrarre tokenizer testuale
text_tok = tokenizer.tokenizer if hasattr(tokenizer, 'tokenizer') else tokenizer
if text_tok.pad_token is None:
    text_tok.pad_token = text_tok.eos_token

# Carica dataset
train_dataset = load_dataset('json', data_files=train_file, split='train')
eval_dataset = load_dataset('json', data_files=eval_file, split='train') if eval_file else None

def format_chatml(example):
    text = text_tok.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

train_dataset = train_dataset.map(format_chatml, num_proc=4)
if eval_dataset:
    eval_dataset = eval_dataset.map(format_chatml, num_proc=4)

# Verifica distribuzione
import json as _json
from collections import Counter
ic = Counter()
with open(train_file) as f:
    for line in f:
        msg = _json.loads(line)
        out = _json.loads(msg['messages'][2]['content'])
        ic[out.get('intent', '?')] += 1

print(f'✅ Dataset pronti!')
print(f'   Train: {len(train_dataset):,} | Eval: {len(eval_dataset) if eval_dataset else 0:,}')
print(f'\n📊 Distribuzione intent:')
for intent, count in sorted(ic.items(), key=lambda x: -x[1]):
    pct = 100 * count / len(train_dataset)
    bar = '█' * int(pct / 2)
    print(f'   {intent:12} {bar} {count:,} ({pct:.1f}%)')

In [ ]:
# ═══ Cell 6: Pre-tokenizzazione + Packing manuale ═══
# CRITICO: packing riduce gli steps da 22h a ~2h su A100!

print('⏳ Pre-tokenizzazione...')
tok_train = train_dataset.map(
    lambda ex: {
        **text_tok(ex['text'], truncation=True, max_length=2048),
        'labels': text_tok(ex['text'], truncation=True, max_length=2048)['input_ids']
    },
    remove_columns=['text', 'messages'],
    num_proc=4,
)

if eval_dataset:
    tok_eval = eval_dataset.map(
        lambda ex: {
            **text_tok(ex['text'], truncation=True, max_length=2048),
            'labels': text_tok(ex['text'], truncation=True, max_length=2048)['input_ids']
        },
        remove_columns=['text', 'messages'],
        num_proc=4,
    )

# ═══ PACKING MANUALE ═══
print('📦 Packing manuale (max_len=2048)...')

def pack_sequences(dataset, max_len=2048):
    eos = text_tok.eos_token_id
    all_ids, all_labels, all_mask = [], [], []
    buf_ids, buf_labels = [], []
    for ex in dataset:
        buf_ids.extend(ex['input_ids'] + [eos])
        buf_labels.extend(ex['labels'] + [eos])
        while len(buf_ids) >= max_len:
            all_ids.append(buf_ids[:max_len])
            all_labels.append(buf_labels[:max_len])
            all_mask.append([1] * max_len)
            buf_ids = buf_ids[max_len:]
            buf_labels = buf_labels[max_len:]
    return Dataset.from_dict({
        'input_ids': all_ids,
        'attention_mask': all_mask,
        'labels': all_labels,
    })

packed_train = pack_sequences(tok_train)
packed_eval = pack_sequences(tok_eval) if eval_dataset else None

ratio = len(train_dataset) / max(len(packed_train), 1)
print(f'\n✅ Packing completato!')
print(f'   {len(train_dataset):,} esempi → {len(packed_train):,} packed sequences')
print(f'   Rapporto: {ratio:.1f}x compressione')
print(f'   (ogni sequence e\' 2048 token, zero padding!)')

In [ ]:
# ═══ Cell 7: Training con completions-only ═══
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

BATCH_SIZE = 8
GRAD_ACCUM = 2
EPOCHS = 3
LR = 2e-4
EVAL_STEPS = 200
SAVE_STEPS = 200

effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(packed_train) // effective_batch

trainer = SFTTrainer(
    model=model,
    tokenizer=text_tok,
    train_dataset=packed_train,
    eval_dataset=packed_eval,
    packing=False,  # gia' fatto manualmente!
    data_collator=DataCollatorForSeq2Seq(tokenizer=text_tok, padding=False),
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type='cosine',
        warmup_steps=50,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        eval_strategy='steps' if packed_eval else 'no',
        eval_steps=EVAL_STEPS if packed_eval else None,
        save_strategy='steps',
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        load_best_model_at_end=True if packed_eval else False,
        metric_for_best_model='eval_loss' if packed_eval else None,
        logging_steps=25,
        report_to='none',
        output_dir='galileo-brain-v9',
        seed=3407,
        remove_unused_columns=False,
    ),
)

total_steps = steps_per_epoch * EPOCHS
print(f'✅ Trainer v9 configurato!')
print(f'   Modello: {MODEL_NAME} (bf16 LoRA r=16)')
print(f'   Batch: {BATCH_SIZE} x {GRAD_ACCUM} = {effective_batch} effective')
print(f'   Steps/epoch: ~{steps_per_epoch:,} | Total: ~{total_steps:,} ({EPOCHS} epochs)')
print(f'   LR: {LR} | Packing: manuale | Completions-only ready')
print(f'\n   Stima: ~1-2h (A100) / ~4-6h (T4)')
print(f'\n🚀 Esegui Cell 8 per avviare!')

In [ ]:
# ═══ Cell 8: TRAINING ═══
import time

print('🚀 Training v9 avviato...')
print(f'   Dataset: {len(packed_train):,} packed sequences')
print(f'   Stima: ~1-2h su A100\n')

start = time.time()
stats = trainer.train()
elapsed = time.time() - start

print(f'\n{"="*60}')
print(f'✅ TRAINING v9 COMPLETATO!')
print(f'   Tempo: {elapsed/60:.1f} min ({elapsed/3600:.1f} ore)')
print(f'   Loss finale: {stats.training_loss:.4f}')
print(f'   Steps: {stats.global_step}')
print(f'   VRAM picco: {torch.cuda.max_memory_allocated()/1024**3:.1f} GB')
print(f'{"="*60}')

In [ ]:
# ═══ Cell 9: Test inferenza rapido ═══
from unsloth import FastLanguageModel
import json

FastLanguageModel.for_inference(model)

# System prompt dal dataset
with open(train_file) as f:
    first = json.loads(f.readline())
system_prompt = first['messages'][0]['content']

test_cases = [
    ('avvia la simulazione', 'action'),
    ('ferma tutto', 'action'),
    ('metti un LED rosso', 'circuit'),
    ('cos\'e\' la legge di Ohm?', 'tutor'),
    ('guarda il mio circuito', 'vision'),
    ('carica il blink', 'navigation'),
    ('scrivi il codice per il LED', 'code'),
    ('come preparo la lezione?', 'teacher'),
    ('misura la tensione', 'action'),
    ('metti la resistenza a 220 ohm', 'action'),
    ('premi il pulsante', 'action'),
    ('giochiamo a detective', 'tutor'),
    ('passa a Scratch', 'action'),
    ('cercami un video', 'navigation'),
    ('gira il potenziometro', 'action'),
    ('evidenzia il LED', 'action'),
    ('avanti col montaggio', 'navigation'),
    ('come faccio un if con i blocchi?', 'code'),
    ('daje fallo anda\'', 'action'),
    ('nn capisco nnt aiuto', 'tutor'),
]

passed = 0
for user_msg, expected in test_cases:
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_msg},
    ]
    inputs = text_tok.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    output = model.generate(
        input_ids=inputs, max_new_tokens=256,
        temperature=0.1, do_sample=True
    )
    response = text_tok.decode(
        output[0][inputs.shape[-1]:], skip_special_tokens=True
    ).strip()
    if '</think>' in response:
        response = response.split('</think>')[-1].strip()
    try:
        parsed = json.loads(response)
        intent = parsed.get('intent', '???')
        ok = '✅' if intent == expected else '❌'
        if intent == expected: passed += 1
        print(f'{ok} [{intent:10}] {user_msg[:50]}')
    except json.JSONDecodeError:
        print(f'❌ [PARSE ERR] {user_msg[:50]}')
        print(f'   Raw: {response[:100]}')

print(f'\n📊 {passed}/{len(test_cases)} ({100*passed/len(test_cases):.0f}%)')

In [ ]:
# ═══ Cell 10: Export GGUF Q5_K_M + Salva su Drive ═══
print('📦 Esportazione GGUF Q5_K_M (qualita\' superiore, ~1.5GB)...')
print('   Questo richiede ~5-10 minuti\n')

OUTPUT_DIR = 'galileo-brain-v9-gguf'

model.save_pretrained_gguf(
    OUTPUT_DIR,
    tokenizer,
    quantization_method='q5_k_m',  # v9: Q5 per qualita' (vs q4 in v7)
)

print('\n✅ GGUF Q5_K_M esportato!')

import os, glob, shutil
gguf_files = glob.glob(f'{OUTPUT_DIR}/*.gguf')
for f in gguf_files:
    size_gb = os.path.getsize(f) / 1024**3
    print(f'   {f}: {size_gb:.2f} GB')

# Salva su Drive
DRIVE_OUT = '/content/drive/MyDrive/galileo-brain-v9-training'
os.makedirs(DRIVE_OUT, exist_ok=True)
for f in gguf_files:
    dest = os.path.join(DRIVE_OUT, os.path.basename(f))
    shutil.copy(f, dest)
    print(f'   → Copiato su Drive: {dest}')

# Copia anche il Modelfile
modelfiles = glob.glob(f'{OUTPUT_DIR}/Modelfile*')
for f in modelfiles:
    dest = os.path.join(DRIVE_OUT, os.path.basename(f))
    shutil.copy(f, dest)

print(f'\n✅ Tutto salvato su Drive: {DRIVE_OUT}')

In [ ]:
# ═══ Cell 11: Istruzioni deploy Mac ═══
print('='*60)
print('🎉 GALILEO BRAIN v9 — TRAINING COMPLETATO!')
print('='*60)
print()
print('Deploy su Mac:')
print('  1. Scarica il GGUF da Google Drive')
print('  2. mkdir -p ~/models/galileo-brain-v9')
print('  3. cp galileo-brain-v9-Q5_K_M.gguf ~/models/galileo-brain-v9/')
print()
print('  4. Crea ~/models/galileo-brain-v9/Modelfile:')
print('     FROM ./galileo-brain-v9-Q5_K_M.gguf')
print('     PARAMETER temperature 0.1')
print('     PARAMETER top_p 0.9')
print('     PARAMETER num_ctx 2048')
print('     PARAMETER stop <|im_end|>')
print('     TEMPLATE """<|im_start|>system')
print('     {{ .System }}<|im_end|>')
print('     <|im_start|>user')
print('     {{ .Prompt }}<|im_end|>')
print('     <|im_start|>assistant')
print('     """')
print()
print('  5. cd ~/models/galileo-brain-v9')
print('  6. ollama create galileo-brain -f Modelfile')
print('  7. ollama run galileo-brain "avvia la simulazione"')
print()
print('v9 improvements vs v7:')
print('  - 126K esempi (vs 86K) — 47% piu\' dati')
print('  - 24/24 action tags (vs 10/16)')
print('  - 7% vision (vs 2.9%)')
print('  - Q5_K_M GGUF — qualita\' superiore')
print('  - LoRA r=16 — piu\' efficiente per routing')